# Clima anual por município (BR-DWGD)

🎯 Objetivo: gerar uma tabela município × ano (1985-2023) com indicadores
de temperatura (média, amplitude, sazonalidade e extremos) e, opcionalmente,
chuva e umidade, a partir das estatísticas zonais diárias do BR-DWGD
(Xavier et al., v3.2.3) calculadas para cada município por Saldanha et al.
(Zenodo, DOI [10.5281/zenodo.13906834](https://doi.org/10.5281/zenodo.13906834)).

Por que ir além da média anual? A literatura sobre desmatamento e clima
local mostra que a perda de vegetação mexe mais com a temperatura MÁXIMA,
com a amplitude térmica, com a estação seca e com os dias extremos do que
com a média anual, que dilui esses efeitos. Ter essas variáveis permite
testar onde o efeito aparece.

**Como usar:** menu *Ambiente de execução > Executar tudo*. No fim, o arquivo `clima_anual_municipios.csv` é baixado automaticamente.

## 1. INSTALAÇÃO, IMPORTS E PARÂMETROS

In [ ]:
# O DuckDB lê arquivos Parquet grandes (~2 GB cada) sem carregar tudo na
# memória, o que evita estourar a RAM do Colab
!pip install -q duckdb

import os
import duckdb
import pandas as pd

ANO_INICIO, ANO_FIM = 1985, 2023   # a série do BR-DWGD termina em mar/2024

# Período de referência para os limiares "relativos" (percentil 90 de cada
# município). Uso os 30 primeiros anos da série, como numa normal climatológica.
REF_INICIO, REF_FIM = 1985, 2014

# Chuva e umidade estavam na lista original de variáveis. Cada uma é mais um
# arquivo de ~2 GB. Se quiser só temperatura, troque para False.
INCLUIR_CHUVA_UMIDADE = True

URL_BASE = 'https://zenodo.org/records/13906834/files'
ARQUIVOS = {'tmax': 'Tmax_3.2.3.parquet', 'tmin': 'Tmin_3.2.3.parquet'}
if INCLUIR_CHUVA_UMIDADE:
    ARQUIVOS.update({'pr': 'pr_3.2.3.parquet', 'rh': 'RH_3.2.3.parquet'})

## 2. DOWNLOAD

In [ ]:
# ⏳ No Colab o download costuma levar poucos minutos por arquivo.
for var, arq in ARQUIVOS.items():
    # --continue retoma um download interrompido em vez de começar do zero, e
    # --tries insiste se o Zenodo oscilar
    if not os.path.exists(arq) or os.path.getsize(arq) < 1.8e9:
        print(f"⏳ Baixando {arq}...")
        !wget -q --show-progress --continue --tries=10 -O {arq} "{URL_BASE}/{arq}?download=1"
    tamanho_gb = os.path.getsize(arq) / 1e9
    print(f"✓ {arq}: {tamanho_gb:.2f} GB")
    # ⚠️ Se o arquivo ficou pequeno, o download falhou no meio (ou o Zenodo
    # devolveu uma página de erro). Melhor parar aqui do que gerar dados errados.
    assert tamanho_gb > 1.5, f"{arq} com {tamanho_gb:.2f} GB: download incompleto, rode a célula de novo"

# Banco DuckDB em DISCO (e não na memória): a tabela diária tem ~80 milhões
# de linhas e, em disco, o DuckDB só traz para a RAM o que está usando.
# ⚠️ Testei na escala real e, com o banco em memória, o cálculo dos
# percentis estourou 6 GB de RAM. Com banco em disco, limite de memória e
# processamento em lotes (seção 5), o uso fica controlado.
if os.path.exists('clima_tmp.duckdb'):
    os.remove('clima_tmp.duckdb')
con = duckdb.connect('clima_tmp.duckdb')
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")

## 3. INSPEÇÃO DA ESTRUTURA E ESCOLHA DA ESTATÍSTICA ZONAL

In [ ]:
# Os arquivos vêm em formato longo: code_muni, date, name, value, onde "name"
# indica a estatística zonal (média, máximo, mínimo, desvio... das células da
# grade dentro do município). Confiro a estrutura em vez de supor.
nomes_estatisticas = {}
for var, arq in ARQUIVOS.items():
    print(f"\n📊 {arq}")
    print(con.sql(f"DESCRIBE SELECT * FROM '{arq}'").df().to_string())
    nomes_estatisticas[var] = con.sql(f"SELECT DISTINCT name FROM '{arq}'").df()['name'].tolist()
    print(f"Estatísticas disponíveis: {nomes_estatisticas[var]}")


def nome_estatistica_media(var):
    """Acha o valor da coluna 'name' que corresponde à média zonal.
    ⚠️ ATENÇÃO METODOLÓGICA: uso a MÉDIA das células do município em todas as
    variáveis. Para a chuva isso também é o certo: a média espacial da chuva
    diária é a lâmina (mm) que caiu no município; a estatística 'sum' somaria
    as células e o valor dependeria do tamanho do município."""
    candidatos = [n for n in nomes_estatisticas[var] if 'mean' in str(n).lower()]
    assert len(candidatos) == 1, f"Esperava 1 estatística de média, achei: {candidatos}"
    return candidatos[0]


def serie_diaria(var):
    """SQL que devolve a série diária (code_muni, date, valor) de uma variável.
    Os CASTs deixam o código funcionando mesmo se a data vier como texto ou o
    código do município vier como texto/decimal."""
    return f"""
        SELECT CAST(code_muni AS BIGINT)  AS code_muni,
               CAST(date AS DATE)         AS date,
               CAST(value AS DOUBLE)      AS {var}
        FROM '{ARQUIVOS[var]}'
        WHERE name = '{nome_estatistica_media(var)}'
          AND YEAR(CAST(date AS DATE)) BETWEEN {ANO_INICIO} AND {ANO_FIM}
    """

## 4. SÉRIE DIÁRIA DE TEMPERATURA (Tmáx e Tmín no mesmo dia)

In [ ]:
# Junto máxima e mínima de cada dia numa tabela só, porque várias métricas
# (amplitude, média diária) precisam das duas ao mesmo tempo. São ~80 milhões
# de linhas; o DuckDB guarda isso de forma compacta e usa o disco se precisar.
print("\n⏳ Montando a série diária de temperatura...")
# FLOAT (4 bytes) em vez de DOUBLE: a precisão de ~7 dígitos sobra para
# temperatura e a tabela fica com metade do tamanho. Ordeno por município
# para que a leitura em lotes (abaixo) pegue blocos contíguos do disco.
con.sql(f"""
    CREATE OR REPLACE TABLE temp_diaria AS
    SELECT x.code_muni, x.date,
           CAST(x.tmax AS FLOAT) AS tmax,
           CAST(n.tmin AS FLOAT) AS tmin
    FROM ({serie_diaria('tmax')}) x
    JOIN ({serie_diaria('tmin')}) n USING (code_muni, date)
    ORDER BY x.code_muni, x.date
""")
print(f"✓ {con.sql('SELECT COUNT(*) FROM temp_diaria').fetchone()[0]:,} dias-município")

## 5. INDICADORES ANUAIS DE TEMPERATURA (em lotes de municípios)

In [ ]:
# Percentis exigem guardar todos os valores do grupo na memória, então
# processo ~500 municípios por vez em vez dos 5.571 de uma só vez.
#
# Limiares relativos de cada município: percentil 90 da Tmáx e da Tmín no
# período de referência.
# ⚠️ ATENÇÃO METODOLÓGICA: um limiar fixo (ex.: 35 °C) quase nunca é atingido
# no Sul e é atingido quase todo dia em partes do Nordeste, o que torna a
# variável pouco comparável entre municípios. O limiar relativo mede "dias
# quentes para aquele lugar" e é o padrão dos índices ETCCDI (TX90p, TN90p).
# Simplificação: uso um percentil por município para o ano todo, e não um
# percentil por dia do calendário como no ETCCDI original.
SQL_INDICADORES = """
    WITH lote AS (
        SELECT *, YEAR(date) AS ano, QUARTER(date) AS trimestre
        FROM temp_diaria
        WHERE code_muni BETWEEN {cod_min} AND {cod_max}
    ),
    limiares AS (
        SELECT code_muni,
               QUANTILE_CONT(tmax, 0.9) AS p90_tmax,
               QUANTILE_CONT(tmin, 0.9) AS p90_tmin
        FROM lote
        WHERE ano BETWEEN {ref_ini} AND {ref_fim}
        GROUP BY code_muni
    )
    SELECT d.code_muni, d.ano,
           COUNT(*)                                   AS n_dias,

           -- Médias anuais
           AVG((d.tmax + d.tmin) / 2)                 AS temp_media_c,
           AVG(d.tmax)                                AS tmax_media_c,
           AVG(d.tmin)                                AS tmin_media_c,
           -- Amplitude térmica diária média: a vegetação amortece o calor do
           -- dia e o resfriamento da noite, então desmatar tende a aumentá-la
           AVG(d.tmax - d.tmin)                       AS amplitude_termica_c,

           -- Média das máximas por trimestre do calendário. O 3º trimestre
           -- (jul-set) é o auge da estação seca na maior parte do Brasil,
           -- quando o efeito da vegetação costuma ser mais forte
           AVG(d.tmax) FILTER (WHERE d.trimestre = 1) AS tmax_jfm_c,
           AVG(d.tmax) FILTER (WHERE d.trimestre = 2) AS tmax_amj_c,
           AVG(d.tmax) FILTER (WHERE d.trimestre = 3) AS tmax_jas_c,
           AVG(d.tmax) FILTER (WHERE d.trimestre = 4) AS tmax_ond_c,

           -- Extremos de calor
           MAX(d.tmax)                                AS tmax_dia_mais_quente_c,
           QUANTILE_CONT(d.tmax, 0.95)                AS tmax_p95_c,
           COUNT(*) FILTER (WHERE d.tmax > l.p90_tmax) AS dias_quentes_p90,
           COUNT(*) FILTER (WHERE d.tmin > l.p90_tmin) AS noites_quentes_p90,
           COUNT(*) FILTER (WHERE d.tmax >= 35)       AS dias_tmax_35c
    FROM lote d
    JOIN limiares l USING (code_muni)
    GROUP BY d.code_muni, d.ano
"""

codigos = con.sql("SELECT DISTINCT code_muni FROM temp_diaria ORDER BY 1").df()['code_muni'].tolist()
TAMANHO_LOTE = 500
partes = []
for i in range(0, len(codigos), TAMANHO_LOTE):
    bloco = codigos[i:i + TAMANHO_LOTE]
    partes.append(con.sql(SQL_INDICADORES.format(
        cod_min=bloco[0], cod_max=bloco[-1], ref_ini=REF_INICIO, ref_fim=REF_FIM)).df())
    print(f"⏳ Indicadores de temperatura: {min(i + TAMANHO_LOTE, len(codigos)):,} de {len(codigos):,} municípios")
temp = pd.concat(partes, ignore_index=True)

## 6. CHUVA E UMIDADE (opcional)

In [ ]:
# ⚠️ ATENÇÃO METODOLÓGICA: no modelo, chuva e umidade devem entrar só como
# teste de robustez. A vegetação também influencia a chuva e a umidade, então
# usá-las como controle pode "roubar" parte do efeito que queremos medir.
if INCLUIR_CHUVA_UMIDADE:
    print("⏳ Calculando chuva e umidade anuais...")
    chuva = con.sql(f"""
        SELECT code_muni, YEAR(date) AS ano,
               COUNT(*)                          AS n_dias_pr,
               SUM(pr)                           AS chuva_anual_mm,
               SUM(pr) FILTER (WHERE QUARTER(date) = 3) AS chuva_jas_mm,
               COUNT(*) FILTER (WHERE pr >= 1)   AS dias_com_chuva
        FROM ({serie_diaria('pr')})
        GROUP BY 1, 2
    """).df()
    umidade = con.sql(f"""
        SELECT code_muni, YEAR(date) AS ano,
               COUNT(*)                          AS n_dias_rh,
               AVG(rh)                           AS umidade_media_pct,
               AVG(rh) FILTER (WHERE QUARTER(date) = 3) AS umidade_jas_pct
        FROM ({serie_diaria('rh')})
        GROUP BY 1, 2
    """).df()
    temp = (temp.merge(chuva, on=['code_muni', 'ano'], how='left')
                .merge(umidade, on=['code_muni', 'ano'], how='left'))

## 7. CONFERÊNCIAS E EXPORTAÇÃO

In [ ]:
# Mantenho só anos completos (365 ou 366 dias). Nas variáveis opcionais, se o
# ano estiver incompleto, apago só aquelas colunas em vez da linha inteira.
n_antes = len(temp)
temp = temp[temp['n_dias'] >= 365]
print(f"✓ Removidas {n_antes - len(temp):,} linhas de anos incompletos de temperatura")
if INCLUIR_CHUVA_UMIDADE:
    temp.loc[temp['n_dias_pr'] < 365, ['chuva_anual_mm', 'chuva_jas_mm', 'dias_com_chuva']] = None
    temp.loc[temp['n_dias_rh'] < 365, ['umidade_media_pct', 'umidade_jas_pct']] = None

temp = (temp.rename(columns={'code_muni': 'geocodigo'})
            .drop(columns=[c for c in temp.columns if c.startswith('n_dias')])
            .sort_values(['geocodigo', 'ano'])
            .round(3))

print("=" * 70)
print(f"📊 Municípios: {temp['geocodigo'].nunique():,} | Anos: {temp['ano'].min()}-{temp['ano'].max()}")
print(temp.drop(columns=['geocodigo', 'ano']).describe().T.round(2).to_string())

import numpy as np

# Conferências de faixa física.
# ⚠️ Em vez de travar o notebook (versão anterior), marco as linhas problemáticas
# como vazias e mostro quais são.

# 1) Quais linhas falharam e por quê. A trava contava como "fora da faixa"
# tanto valores absurdos quanto valores vazios (NaN), que aparecem quando o
# município não tem célula da grade do BR-DWGD (ex.: ilhas oceânicas).
problema = ~temp['temp_media_c'].between(5, 35) | ~(temp['tmax_media_c'] > temp['tmin_media_c'])
fora = temp[problema]
print(f"⚠️ Linhas com problema: {len(fora):,} de {len(temp):,} | municípios afetados: {fora['geocodigo'].nunique()}")
print(f"   - sem valor (NaN): {fora['temp_media_c'].isna().sum():,}")
print(f"   - com valor fora de 5-35 °C ou Tmáx <= Tmín: {fora['temp_media_c'].notna().sum():,}")
print("\nMunicípios afetados (nº de anos com problema, primeiro e último ano):")
print(fora.groupby('geocodigo')['ano'].agg(['count', 'min', 'max']).to_string())
print("\nExemplos:")
print(fora[['geocodigo', 'ano', 'temp_media_c', 'tmax_media_c', 'tmin_media_c']].head(10).to_string(index=False))

# 2) Nessas linhas, todos os indicadores de temperatura viram vazio.
# ⚠️ ATENÇÃO METODOLÓGICA: sem isso, as contagens (dias quentes, dias acima
# de 35 °C) apareceriam como 0, que é diferente de "sem dado" e distorceria
# o modelo. Chuva e umidade têm checagem própria logo abaixo.
cols_temp = [c for c in temp.columns if c.startswith(
    ('temp_', 'tmax_', 'tmin_', 'amplitude', 'dias_quentes', 'noites_quentes', 'dias_tmax'))]
temp.loc[problema, cols_temp] = np.nan

if 'chuva_anual_mm' in temp.columns:
    chuva_ruim = ~temp['chuva_anual_mm'].between(0, 6000)
    temp.loc[chuva_ruim, ['chuva_anual_mm', 'chuva_jas_mm', 'dias_com_chuva']] = np.nan
    umid_ruim = ~temp['umidade_media_pct'].between(0, 100)
    temp.loc[umid_ruim, ['umidade_media_pct', 'umidade_jas_pct']] = np.nan
    print(f"\n⚠️ Chuva sem valor ou fora de 0-6.000 mm: {chuva_ruim.sum():,} linhas | "
          f"umidade fora de 0-100%: {umid_ruim.sum():,} linhas")

# 3) Conferências no que sobrou e exportação
ok = temp['temp_media_c'].notna()
print(f"\n✓ Linhas válidas de temperatura: {ok.sum():,} ({ok.mean():.1%}) | "
      f"municípios com algum dado: {temp.loc[ok, 'geocodigo'].nunique():,}")
print(f"✓ Faixa da temperatura média: {temp['temp_media_c'].min():.1f} a {temp['temp_media_c'].max():.1f} °C")
ref = temp[temp['ano'].between(REF_INICIO, REF_FIM) & ok]
print(f"✓ Dias quentes (p90) no período de referência: {ref['dias_quentes_p90'].mean():.1f} por ano (esperado ≈ 36,5)")
print(temp[temp['geocodigo'] == 3548906].tail(3).set_index('ano').drop(columns='geocodigo').T.to_string())

temp.to_csv('clima_anual_municipios.csv', index=False)
print(f"💾 Salvo: clima_anual_municipios.csv ({os.path.getsize('clima_anual_municipios.csv') / 1e6:.1f} MB)")

# Fecho e apago o banco temporário (pode passar de 1 GB no disco do Colab)
try:
    con.close()
    os.remove('clima_tmp.duckdb')
except Exception:
    pass

from google.colab import files
files.download('clima_anual_municipios.csv')
